In [1]:
import os
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict

load_dotenv()

True

In [6]:
llm = ChatOllama(model='gemma4:31b-cloud')


### Workflow (1): START >> LLM Q/A >> END

In [7]:
class LLMState(TypedDict):

    question: str
    answer: str

In [8]:
## Define function

def llm_qa(state: LLMState)->LLMState:

    question = state['question']

    prompt = f"Answer the following quesion: {question}"

    answer = llm.invoke(prompt).content

    state['answer'] = answer

    return state

In [11]:
# graph
graph = StateGraph(LLMState)

# add node
graph.add_node('llm_qa',llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa',END)

# compile
workflow = graph.compile()

In [12]:
initial_state = {"question":"How far is the moon from the earth?"}

final_state = workflow.invoke(initial_state)

final_state['answer']

'The average distance from the Earth to the Moon is approximately **238,855 miles** (384,400 kilometers).\n\nBecause the Moon follows an elliptical (oval-shaped) orbit rather than a perfect circle, this distance changes constantly:\n\n*   **Perigee (Closest approach):** About **225,623 miles** (363,104 km).\n*   **Apogee (Farthest point):** About **252,088 miles** (405,696 km).'